# EDA — Exploring a New Dataset

## Notebook goal

In this notebook, we explore cars dataset before cleaning or modeling.

The goal is to understand:

- what the dataset contains,
- how the columns are structured,
- what types of values appear in the columns,
- where missing or inconsistent values exist,
- which problems need to be fixed during the cleaning phase.


## 1. Intro

Before training a regression model, we need to understand the data.

A raw dataset can contain many problems:

- inconsistent column names,
- missing values,
- numbers stored as text,
- categorical values written inconsistently,
- target values stored in the wrong format.

If we train a model before checking these issues, the model may fail, or it may learn from unreliable data.

In this notebook, the goal is to inspect the dataset and identify what needs to be cleaned later.

## 2. Import libraries

We start by importing the libraries needed for basic data exploration.

In [59]:
from pathlib import Path

import pandas as pd

## 3. Define the data path

The path below assumes the following project structure:

```text
TASK 4/
├── data/
│   └── cars.csv
└── notebooks/
    └── 01_eda_cars_dataset.ipynb
```


In [60]:
DATA_PATH = Path("../data/cars.csv")

DATA_PATH

PosixPath('../data/cars.csv')

## 4. Load the dataset

We load the raw CSV file into a pandas DataFrame.

In [61]:
df = pd.read_csv(DATA_PATH)

## 5. First look at the data

The first few rows give us an initial impression of the dataset.

In [62]:
df.head()

,make,model,priceUSD,year,condition,mileage(kilometers),fuel_type,volume(cm3),color,transmission,drive_unit,segment
0,mazda,2,5500,2008,with mileage,162000.0,petrol,1500.0,burgundy,mechanics,front-wheel drive,B
1,mazda,2,5350,2009,with mileage,120000.0,petrol,1300.0,black,mechanics,front-wheel drive,B
2,mazda,2,7000,2009,with mileage,61000.0,petrol,1500.0,silver,auto,front-wheel drive,B
3,mazda,2,3300,2003,with mileage,265000.0,diesel,1400.0,white,mechanics,front-wheel drive,B
4,mazda,2,5200,2008,with mileage,97183.0,diesel,1400.0,gray,mechanics,front-wheel drive,B


It is also useful to inspect a few random rows, because the first rows are not always representative.

In [63]:
df.sample(5, random_state=42)

,make,model,priceUSD,year,condition,mileage(kilometers),fuel_type,volume(cm3),color,transmission,drive_unit,segment
19270,mitsubishi,carisma,2050,1999,with mileage,250000.0,petrol,1800.0,blue,mechanics,front-wheel drive,M
25927,ford,fusion,4500,2006,with mileage,160000.0,petrol,1400.0,burgundy,mechanics,front-wheel drive,M
23388,mercedes-benz,e-klass,3500,1999,with mileage,485000.0,diesel,2900.0,blue,mechanics,rear drive,E
53189,bmw,x3,21000,2013,with mileage,159000.0,diesel,2000.0,black,auto,NaN,J
34058,renault,megane,3990,2002,with mileage,331700.0,diesel,1900.0,other,mechanics,front-wheel drive,C


Check the summary statistics. Check the min and max values of priceUSD, year, mileage(kilometers) and volumen(cm3).

In [64]:
df.describe()

,priceUSD,year,mileage(kilometers),volume(cm3)
count,56244.000000,56244.000000,5.624400e+04,56197.000000
mean,7415.456440,2003.454840,2.443956e+05,2104.860615
std,8316.959261,8.144247,3.210307e+05,959.201633
min,48.000000,1910.000000,0.000000e+00,500.000000
25%,2350.000000,1998.000000,1.370000e+05,1600.000000
50%,5350.000000,2004.000000,2.285000e+05,1996.000000
75%,9807.500000,2010.000000,3.100000e+05,2300.000000
max,235235.000000,2019.000000,9.999999e+06,20000.000000


## 6. Basic dataset structure

Now we check the basic structure of the dataset.

In [65]:
df.shape

(56244, 12)

In [66]:
print(f"Number of rows: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")

Number of rows: 56244
Number of columns: 12


Next, we inspect the column names.

In [67]:
df.columns.tolist()

['make',
 'model',
 'priceUSD',
 'year',
 'condition',
 'mileage(kilometers)',
 'fuel_type',
 'volume(cm3)',
 'color',
 'transmission',
 'drive_unit',
 'segment']

The `info()` method shows column names, detected data types, and the number of non-missing values.

In [68]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 56244 entries, 0 to 56243
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   make                 56244 non-null  str    
 1   model                56244 non-null  str    
 2   priceUSD             56244 non-null  int64  
 3   year                 56244 non-null  int64  
 4   condition            56244 non-null  str    
 5   mileage(kilometers)  56244 non-null  float64
 6   fuel_type            56244 non-null  str    
 7   volume(cm3)          56197 non-null  float64
 8   color                56244 non-null  str    
 9   transmission         56244 non-null  str    
 10  drive_unit           54339 non-null  str    
 11  segment              50953 non-null  str    
dtypes: float64(2), int64(2), str(8)
memory usage: 5.1 MB


From this initial overview, we can already identify several issues that will need to be handled during data cleaning:

- column names should be standardized so they are easier to use in Python code,
- date and time columns were not detected as proper date/time types,
- missing values appear in many columns,
- the target column needs to be inspected more carefully before it can be used for regression.

These observations give us the first cleaning tasks that will be addressed later in the project.

## 7. Understanding the meaning of columns

Before cleaning the data, we need to understand what the columns represent.

We can roughly group the columns into:

- identifier columns,
- car description,
- numerical columns,
- target column.

This helps us decide how each group of columns should be inspected.

Typical identifier columns are:

In [69]:
identifier_columns = [
    "make",
    "model",
]

df[identifier_columns].head()

,make,model
0,mazda,2
1,mazda,2
2,mazda,2
3,mazda,2
4,mazda,2


Description columns are:

In [70]:
description_columns = [
    "condition",
    "fuel_type",
    "color",
    "transmission",
    "drive_unit",
    "segment",
    
]

df[description_columns].head()

,condition,fuel_type,color,transmission,drive_unit,segment
0,with mileage,petrol,burgundy,mechanics,front-wheel drive,B
1,with mileage,petrol,black,mechanics,front-wheel drive,B
2,with mileage,petrol,silver,auto,front-wheel drive,B
3,with mileage,diesel,white,mechanics,front-wheel drive,B
4,with mileage,diesel,gray,mechanics,front-wheel drive,B


Typical numerical columns are:

In [71]:
numerical_columns = [
    "year",
    "mileage(kilometers)",
    "volume(cm3)",
]

df[numerical_columns].head()

,year,mileage(kilometers),volume(cm3)
0,2008,162000.0,1500.0
1,2009,120000.0,1300.0
2,2009,61000.0,1500.0
3,2003,265000.0,1400.0
4,2008,97183.0,1400.0


The target column is:

In [72]:
target_column = "priceUSD"

df[target_column].head()

0    5500
1    5350
2    7000
3    3300
4    5200
Name: priceUSD, dtype: int64

## 8. Target column inspection

The target column contains the value we want to predict.

For this project, the target is delivery time in minutes.

In [73]:
df[target_column].head(10)

0    5500
1    5350
2    7000
3    3300
4    5200
5    3400
6    5000
7    7300
8    6400
9    6132
Name: priceUSD, dtype: int64

We check the detected data type.

In [74]:
df[target_column].dtype

dtype('int64')

We also inspect a few unique values to understand how the target is written.

In [75]:
df[target_column].unique()[:10]

array([5500, 5350, 7000, 3300, 5200, 3400, 5000, 7300, 6400, 6132])

The target column is already stored in a numeric format.

Since pandas detected this column as `int64`, the values are already integers and can be used as numeric target values for a regression model.

Because of that, the target column does not require text extraction or conversion in the cleaning phase. However, we should still check whether it contains missing values or unusually small or large delivery times.

## 9. Missing values

First, we check real missing values detected by pandas.

In [76]:
missing_values = df.isna().sum()

missing_values[missing_values > 0]

volume(cm3)      47
drive_unit     1905
segment        5291
dtype: int64

Raw data can also contain missing-like text values, such as `"NaN"`, `"null"`, `"None"`, or empty strings.

These values may look like regular text, but they usually mean that the value is missing.

In [77]:
missing_like_values = ["NaN", "nan", "NULL", "null", "None", "none", "", " "]

for column in df.columns:
    if df[column].dtype == "object":
        count = df[column].astype(str).str.strip().isin(missing_like_values).sum()

        if count > 0:
            print(f"{column}: {count}")

A certain number of columns contain missing values.

Later, during the cleaning and preprocessing phases, we will decide how to handle them. In some cases, rows with missing values may be removed completely. In other cases, we may use an imputation strategy and fill missing values with appropriate replacement values.

The important thing in this notebook is to identify where missing values appear, so we can decide how to handle them in the next steps.

## 10. Numeric column inspection

Next, we inspect columns that should represent numeric values.

In [78]:
numeric_columns = [
    "year",
    "mileage(kilometers)",
    "volume(cm3)",
    ]

First, we check their detected data types.

In [79]:
df[numeric_columns].dtypes

year                     int64
mileage(kilometers)    float64
volume(cm3)            float64
dtype: object

All selected numeric columns were detected as numeric types.

The only irregularity we can notice is that some columns use a `float` type even though their values appear to be whole numbers. This is most likely caused by missing values, because pandas often converts integer columns with missing values into floating-point columns.

Because of that, these columns do not require text-to-number conversion, but we will still need to decide how to handle missing values later in the cleaning and preprocessing phases.

## 11. Categorical column inspection

Now we inspect columns that contain categorical values.

In [80]:
categorical_columns = [
    "make",
    "model",
    "condition",
    "fuel_type",
    "color",
    "transmission",
    "drive_unit",
    "segment",

]

In [81]:
for col in df.select_dtypes(include="object"):
    print(df[col].value_counts())

make
volkswagen    6861
audi          4030
bmw           4013
opel          3779
renault       3713
              ... 
trabant          1
jac              1
asia             1
tagaz            1
saipa            1
Name: count, Length: 96, dtype: int64
model
passat      2086
5-seriya    1476
a6          1276
golf        1070
astra       1013
            ... 
xb             1
xc40           1
xjs            1
xt5            1
z3             1
Name: count, Length: 1034, dtype: int64
condition
with mileage    55278
with damage       512
for parts         454
Name: count, dtype: int64
fuel_type
petrol        36405
diesel        19792
electrocar       47
Name: count, dtype: int64
color
black       12385
silver      10075
blue         8083
gray         5807
white        5292
green        3911
other        3397
red          2744
burgundy     2026
brown        1349
purple        625
yellow        317
orange        233
Name: count, dtype: int64
transmission
mechanics    36056
auto         20188


/tmp/ipykernel_14957/3344944030.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object"):


First, we check how many unique values each categorical column contains.

In [82]:
for column in categorical_columns:
    print(column)
    print(df[column].nunique(dropna=False))
    print("-" * 40)

make
96
----------------------------------------
model
1034
----------------------------------------
condition
3
----------------------------------------
fuel_type
3
----------------------------------------
color
13
----------------------------------------
transmission
2
----------------------------------------
drive_unit
5
----------------------------------------
segment
10
----------------------------------------


Then we inspect the actual values and their frequencies.

In [83]:
for column in categorical_columns:
    print(df[column].value_counts(dropna=False))
    print("-" * 40)

make
volkswagen    6861
audi          4030
bmw           4013
opel          3779
renault       3713
              ... 
trabant          1
jac              1
asia             1
tagaz            1
saipa            1
Name: count, Length: 96, dtype: int64
----------------------------------------
model
passat      2086
5-seriya    1476
a6          1276
golf        1070
astra       1013
            ... 
xb             1
xc40           1
xjs            1
xt5            1
z3             1
Name: count, Length: 1034, dtype: int64
----------------------------------------
condition
with mileage    55278
with damage       512
for parts         454
Name: count, dtype: int64
----------------------------------------
fuel_type
petrol        36405
diesel        19792
electrocar       47
Name: count, dtype: int64
----------------------------------------
color
black       12385
silver      10075
blue         8083
gray         5807
white        5292
green        3911
other        3397
red          2744
bur

Based on this initial inspection, the categorical columns look consistent.

The expected categorical columns were correctly identified as categorical values, and there are no obvious cases where the same category appears in multiple different forms.

At this stage, we do not see major issues such as duplicated category labels, inconsistent capitalization, or unnecessary variations of the same value. These columns will still need to be encoded later, but they do not appear to require significant category cleaning.

## 13. Duplicate rows

Duplicate rows can distort analysis and model training.

We check whether complete duplicate rows exist.

In [84]:
duplicate_rows = df.duplicated()

print("Duplicate rows:", duplicate_rows.sum())

Duplicate rows: 87


In [85]:
df.loc[duplicate_rows].head()

,make,model,priceUSD,year,condition,mileage(kilometers),fuel_type,volume(cm3),color,transmission,drive_unit,segment
1930,audi,100,1280,1991,with mileage,305000.0,petrol,2300.0,red,mechanics,front-wheel drive,E
2054,audi,100,2000,1988,with mileage,350000.0,petrol,2300.0,blue,mechanics,front-wheel drive,E
2140,audi,100,1100,1989,with mileage,350000.0,petrol,2300.0,burgundy,mechanics,front-wheel drive,E
2179,peugeot,106,700,2000,with mileage,27000.0,diesel,1500.0,blue,mechanics,front-wheel drive,B
4334,peugeot,406,2200,1995,with mileage,330000.0,petrol,1800.0,gray,mechanics,front-wheel drive,D


The check shows that there are 87 complete duplicate rows in the dataset.

## 14. Extreme values
- Check the numebr of cars manufactured between year 0 and 1980, as they are considered old cars.
- Verify if there are cars with price below zero.

In [86]:
extreme_years = df["year"].between(0,1980)
print("Extreme year values:", extreme_years.sum())

Extreme year values: 301


In [87]:
extreme_price = df[target_column] < 0
print("Extreme price values:", extreme_price.sum())

Extreme price values: 0


## 15. Final EDA Findings and Cleaning Plan

Based on the exploratory analysis, we identified the following tasks that need to be handled before model training:

- Standardize column names so they are easier and safer to use in Python code.
- Check the target column for missing values.
- Handle missing values in several columns.  
- Review numeric columns that use a `float` type even though they contain whole-number values.  

The result of the cleaning phase should be a cleaner and more reliable version of the dataset, ready for the next stages of the machine learning workflow.